In [24]:
%reset -f
import arcpy
arcpy.ResetEnvironments()
import os
from arcpy.sa import SetNull

# check out Spatial Analyst license
arcpy.CheckOutExtension("Spatial")

'CheckedOut'

In [25]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Input raw US Caribbean 10m raster + GEBCO raster output from "RAW_GEBCO_MOSAIC_g4326" mosaic
usc = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Raw_data\US_Caribbean_Bathymetry_raw.gdb\US_Caribbean_10m"
gbc = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb\GBC15a_bathy_entireEastCarib_withland_g4326"

# Output USC and GBC rasters with land (value > 0) removed
usc_noland = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\USC10m_bathy_p26920_v6642"
gbc_noland = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb\GBC15a_bathy_entireEastCarib_g4326"

In [26]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun
arcpy.env.compression = "NONE"      # no compression on output rasters

In [27]:
# ── INSPECT RASTER DATA BEFORE REMOVING LAND ─────────────────────────────────────────────────────────

# Function to print all info (number of bands, pixel type, min, max, cell size, coordinate systems) for each raster
def print_raster_info(raster):

    # get number of bands info
    d = arcpy.Describe(raster)
    # get pixel type/min/max/cell size info (r = arcpy.Raster() loads raster dataset into a Raster object so arcpy can access stats)
    r = arcpy.Raster(raster)
    # get spatial reference info
    sr = d.spatialReference
    vcs = sr.VCS.name if sr.VCS else "NONE"

    # info print block for raster
    print(f"\n---------------- {os.path.basename(raster)} ----------------: \n")
    print(f"   Number bands = {d.bandCount}")
    print(f"   Pixel type = {r.pixelType}")
    print(f"   ⬇️ Min = {r.minimum}")
    print(f"   ⬆️ Max = {r.maximum}")
    print(f"   📐 Cell size = {r.meanCellWidth}")
    print(f"   Horizontal SR: {sr.name}")
    print(f"   Vertical SR: {vcs}")
    print(f"\n")


# Calculate Statistics (Data Management Tools): (default X/Y skip factor = 1; skip raster if stats already calculated)
for stats_path1 in [gbc, usc]:
    arcpy.management.CalculateStatistics(
        in_raster_dataset=stats_path1,
        skip_existing="SKIP_EXISTING"
    )

# Call function to print info for each input raster
print_raster_info(gbc)
print_raster_info(usc)


---------------- GBC15a_bathy_entireEastCarib_withland_g4326 ----------------: 

   Number bands = 1
   Pixel type = S16
   ⬇️ Min = -8733.0
   ⬆️ Max = 5611.0
   📐 Cell size = 0.0041666666666666605
   Horizontal SR: GCS_WGS_1984
   Vertical SR: NONE



---------------- US_Caribbean_10m ----------------: 

   Number bands = 1
   Pixel type = F32
   ⬇️ Min = -6239.16943359375
   ⬆️ Max = 1335.271240234375
   📐 Cell size = 10.0
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09 height




In [28]:
# ── 1. SET RASTER DATA LAND VALUES (> 0) TO NULL ─────────────────────────────────────────────────────

# Set Null (Spatial Analyst Tools): Function
def remove_land(in_raster, out_path):
    # snap raster to input raster grid so output aligns to same grid
    arcpy.env.snapRaster = in_raster

    try:
        result = SetNull(
            # input raster with cells that will be evaluated by the where_clause condition
            in_conditional_raster=in_raster,
            # data source to populate cells in output raster where cells in input raster have values <= 0 (condition=FALSE), these are cells with water depths
            in_false_raster_or_constant=in_raster,
            # set cells in output raster to Null where cells in input raster have values > 0 (condition=TRUE), these are cells with land elevations
            where_clause="VALUE > 0"
        )
        # save output as new raster
        result.save(out_path)
        print(f"✅ Exported new raster with land removed: {out_path} \n")

    # print warning if tool failed
    except Exception as e:
        print(f"❌[FAILURE] Set Null tool failed for {os.path.basename(out_path)}: {e} \n")


# Call function to remove land (values > 0) from input USC and GBC rasters
remove_land(gbc, gbc_noland)
remove_land(usc, usc_noland)

# Clear settings in environment for subsequent blocks
arcpy.ClearEnvironment("snapRaster")

✅ Exported new raster with land removed: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb\GBC15a_bathy_entireEastCarib_g4326 

✅ Exported new raster with land removed: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\USC10m_bathy_p26920_v6642 



In [29]:
# ── INSPECT RASTER DATA AFTER REMOVING LAND ──────────────────────────────────────────────────────────

# Calculate Statistics (Data Management Tools): (default X/Y skip factor = 1)
for stats_path2 in [gbc_noland, usc_noland]:
    arcpy.management.CalculateStatistics(in_raster_dataset=stats_path2)

# Call function to print info for each output raster
print_raster_info(gbc_noland)
print_raster_info(usc_noland)


---------------- GBC15a_bathy_entireEastCarib_g4326 ----------------: 

   Number bands = 1
   Pixel type = S16
   ⬇️ Min = -8733.0
   ⬆️ Max = 0.0
   📐 Cell size = 0.0041666666666666605
   Horizontal SR: GCS_WGS_1984
   Vertical SR: NONE



---------------- USC10m_bathy_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = F32
   ⬇️ Min = -6239.16943359375
   ⬆️ Max = 0.0
   📐 Cell size = 10.0
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09 height




In [30]:
# ── FIX USC VERTICAL COORDINATE SYSTEM ───────────────────────────────────────────────────────────────

# USC raster VCS is stored as custom string "VIVD09 height" (space) instead of registered "VIVD09_height"

# NAD 1983 UTM Zone 20N (horizontal) + VIVD09 height (vertical)
usc_sr = arcpy.SpatialReference(26920, 6642)

# Overwrite USC raster spatial reference definition in place (** does NOT reproject or alter pixel values)
arcpy.management.DefineProjection(usc_noland, usc_sr)

# Calculate Statistics (Data Management Tools): (default X/Y skip factor = 1)
arcpy.management.CalculateStatistics(
    in_raster_dataset=usc_noland,
    skip_existing="SKIP_EXISTING"
)

# Call function to print info for USC raster
print_raster_info(usc_noland)


---------------- USC10m_bathy_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = F32
   ⬇️ Min = -6239.16943359375
   ⬆️ Max = 0.0
   📐 Cell size = 10.0
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height


